# 2단계 마무리 — 학습률 결론 (후보 4개 × 시드 3개)

2단계 훑어보기(시드 0)에서 옵티마이저마다 최적 학습률이 10배씩 달랐고(Adam 1e-4 · SGD+momentum 1e-3 · SGD 1e-2) 두 프레임워크가 똑같이 나왔다. 다만 시드가 하나라서 가까운 후보끼리의 우열은 읽을 수 없었다. Adam 1e-4가 1e-3보다 val acc가 1.6~2.2%p 높았지만, 같은 설정(Adam 1e-3, 시드 0)을 1단계에서 돌렸을 때와 2단계에서 돌렸을 때의 Keras val acc가 1.6%p 달랐다 (43.1 → 41.5%). 이 노트북은 **후보 4개에 시드 1·2를 더해 3시드로 만들고, 학습률 단계의 결론을 낸다.**

| | 값 |
|---|---|
| 프레임워크 | Keras, PyTorch |
| 후보 (옵티마이저 · 학습률) | Adam · 1e-4, Adam · 1e-3 (현재 기준선), SGD+momentum 0.9 · 1e-3, SGD · 1e-2 |
| 시드 | 0, 1, 2 — **시드 0은 2단계 결과(Drive의 `food11_v2_lr.csv`)를 그대로 쓰고, 1·2만 새로 돈다** |
| 실행 횟수 | 4 × 2(시드 1·2) × 2 = **16번** |
| 그 밖의 조건 | 2단계와 같다: He 초기화, 2 Conv(32·64) + Dropout 0.5 + Dense(11), batch 32, 5 epoch, 증강 없음, 학습률 고정(스케줄러 없음) |
| 소요 시간 | 대략 56분 (Keras 8번 ≈ 20분, PyTorch 8번 ≈ 36분) |

**후보를 이렇게 고른 이유**: 2단계 시드 0에서 옵티마이저마다 val 기준 최적이었던 학습률(val acc와 val loss 최저가 두 프레임워크에서 일치)에, 현재 기준선인 Adam 1e-3을 더했다. 반 자릿수(3e-4 · 3e-3 등)는 보지 않는다. 옆 자릿수끼리의 차이가 Adam은 2%p 안팎, SGD 계열도 최대 5%p라서 그 사이를 쪼개 얻을 이득은 더 작을 것으로 보기 때문이다 (추정).

**이 노트북이 보지 않는 것**
- SGD+momentum과 SGD는 후보가 하나씩이라 이웃 자릿수와 3시드로 비교하지 않는다 (2단계 시드 0의 자릿수 결과를 따른다)
- Adam 1e-4보다 낮은 학습률(3e-5 등)은 보지 않는다. 1e-4가 격자 끝이라는 한계는 남는다
- 5에폭 고정이다. 최저 에폭이 5인 후보는 더 길게 돌리면 결과가 달라질 수 있다

## 결론을 내는 규칙 (결과를 보기 전에 정해 둔다)

v1에서는 Keras AdamW가 4%p 해롭다는 결론이 재실행에서 편차로 밝혀져 빠졌다. 한 번 본 차이를 효과로 믿지 않으려고, "무엇이 나오면 어떻게 결론 낼지"를 미리 적어 둔다. 셀 4가 이 규칙을 그대로 적용해서 출력한다.

**같은 시드끼리 뺀다.** 같은 시드는 초기 가중치와 데이터 순서가 같아서 시드 때문에 생기는 차이가 상당 부분 상쇄된다 (시드 반복 단계에서도 그렇게 했다). 차이가 **"뚜렷함"**이려면 다음 두 가지를 모두 만족해야 한다.

1. 시드 전부(3개)에서 차이의 방향이 같다
2. 평균 차이의 절댓값이 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 크다

**규칙 1 — Adam 기준선 학습률을 1e-3에서 1e-4로 옮길까.** val acc와 val loss 최저 **둘 다**, **두 프레임워크 모두**에서 1e-4가 "뚜렷하게" 앞설 때만, 즉 4칸 전부에서 앞설 때만 옮긴다. 하나라도 아니면 1e-3을 유지한다. 반대로 4칸 전부에서 1e-3이 뚜렷하게 앞서면 "1e-3 확정"이다. 이렇게 엄격하게 잡은 이유는 기준선을 옮기면 뒤의 모든 단계(정규화 등)가 그 위에서 돌기 때문이다. 우연으로 옮기는 쪽이 놓치는 쪽보다 비싸다.

**규칙 2 — 옵티마이저 비교.** 옵티마이저마다 최적 후보끼리(Adam은 두 후보 중 3시드 평균 val acc가 높은 쪽) val acc에 같은 기준을 적용해 "구분됨 / 구분 안 됨"을 표시한다. 결론에는 구분되는 것만 적는다. 고르는 건 val이고 test는 보고용이다.

### 예상 (가설 — 결과로 확인한다)

1. **Adam 1e-4 대 1e-3은 규칙 1을 통과하지 못할 것이다 (기준선 1e-3 유지).** 시드 0에서 val acc 차이가 재실행 흔들림과 같은 크기였다.
2. **val loss 최저는 1e-4 쪽이 낮게 나올 것이다.** 시드 0에서 두 프레임워크가 같은 방향이었다 (1.65 대 1.71, 1.67 대 1.77). 시드 3개에서도 같은 방향이면 "1e-3은 과적합이 일찍 시작해서 바닥이 얕다"는 설명이 힘을 얻는다. 다만 규칙 1은 val acc까지 요구하므로 이것만으로 기준선을 옮기지는 않는다.
3. **Adam과 SGD+momentum은 구분되지 않고, SGD는 둘보다 낮을 것이다.** 시드 0에서 SGD가 6~10%p 낮았다. 시드 3개에서도 "뚜렷함"이 유지되는지 본다.
4. **val loss 최저 에폭은 Adam 1e-4가 5(끝날 때도 내려가는 중), Adam 1e-3이 2~3에폭일 것이다** (시드 0 패턴).
5. 두 프레임워크에서 같은 방향이 나올 것이다.

## 실행 방법

위에서부터 셀을 차례로 돌린다 (셀 0 → 1 → 2 → 3 → 4).

- **Drive(`MyDrive`)에 2단계의 `food11_v2_lr.csv`가 있어야 한다.** 이 파일을 지우지 말 것. 셀 1의 `[계획]` 출력에서 각 프레임워크가 "이미 4번 · 남은 8번"으로 나오면 정상이다. 파일이 없으면 시드 0도 새로 돌게 된다 (+약 28분).
- 새 결과는 같은 CSV 파일에 한 줄씩 쌓인다. **런타임이 끊겨도 남고**, 다시 돌리면 끝난 실행은 건너뛰고 이어서 한다.
- 셀 2(Keras)와 셀 3(PyTorch)은 서로 독립이다. 셀 4(요약)는 있는 결과만으로 만든다. 다만 규칙 판정은 두 프레임워크의 시드 3개씩이 모두 있어야 나오고, 모자라면 "판정 보류"로 나온다.
- 각 셀의 처음에 "이미 있음 — 건너뜀"이 4번 나오는 건 정상이다 (시드 0). 그다음 실행 하나마다 시작 줄 → 에폭 줄 5개 → 끝 줄이 나온다. 에폭 줄은 Keras 약 30초, PyTorch 약 55초 간격이다.
- **셀 4의 출력을 통째로 붙여 주면 된다.**

## 어떻게 읽을까

셀 4는 출력 3개와 그래프 1개를 만든다: `[결과표]`(프레임워크마다) · `[학습률 결론 1]` · `[옵티마이저 비교]` · 그래프 `lr_seeds.png`.

- **결과표**는 평균 [최소~최대]다. 최소~최대가 서로 겹치는 칸은 3시드로도 가려지지 않는 칸이다.
- **학습률 결론 1**의 마지막 `-> 결론` 줄이 규칙 1의 결과다. 칸마다 시드별 차이와 시드 범위가 같이 나오므로 왜 그렇게 판정됐는지 확인할 수 있다.
- **옵티마이저 비교**는 "구분됨"으로 나온 것만 결론으로 적는다.
- 그래프에서 점 하나가 시드 하나이고, 같은 시드는 같은 가로 위치에 있다 (왼쪽부터 시드 0 · 1 · 2). 짧은 가로선이 평균이다.
- 결과표의 비고에 `최저 에폭=마지막`이 있으면 5에폭이 끝날 때도 val loss가 내려가는 중이었다는 뜻이다. 그 후보는 5에폭 기준으로 불리하게 평가됐을 수 있다.

## 셀 0 — 데이터 복사 (최초 1회, 런타임 끊기면 다시)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Drive에서 직접 읽으면 학습이 수십 배 느려져서 로컬로 복사한다
if not os.path.isdir("/content/food11"):
    !cp -r /content/drive/MyDrive/food11 /content/food11

## 셀 1 — 공통 설정, 기록 함수, 실행 계획

In [ ]:
import csv
import gc
import json
import os
import time

import numpy as np

DATA_DIR = "/content/food11"
IMG, BATCH, EPOCHS = 128, 32, 5
NUM_CLASSES = 11
N_PARAMS = 653_003           # 두 프레임워크 모두 이 값이어야 같은 모델이다

# 이 단계에서 보는 후보: 2단계 시드 0 훑기에서 옵티마이저마다 val 기준 최적이었던 학습률
# + 현재 기준선(Adam 1e-3). 나머지(He 초기화, 구조, batch 32, 5 epoch, 증강 없음)는 2단계 그대로다.
CANDIDATES = [("adam", 1e-4), ("adam", 1e-3),            # 1e-3 = 현재 기준선
              ("sgd_momentum", 1e-3), ("sgd", 1e-2)]      # sgd_momentum = SGD + momentum 0.9
SEEDS = (0, 1, 2)            # 시드 0은 2단계 결과가 이미 있어서 건너뛰고 1·2만 돈다
MIN_PER_RUN = {"keras": 2.5, "torch": 4.5}      # 실행 하나에 걸리는 시간 (분, 어림값)

# 2단계와 같은 파일이다. 시드 0이 여기 들어 있고, 새 결과도 여기에 한 줄씩 쌓는다.
# 런타임이 끊겨도 남고, 다시 돌리면 끝난 실행은 건너뛴다. 이 파일을 지우지 말 것.
RESULTS = "/content/drive/MyDrive/food11_v2_lr.csv"
FIELDS = ["framework", "optimizer", "lr", "seed", "n_params", "minutes",
          "train_acc", "train_loss", "val_acc", "val_loss",    # 에폭별 값 (JSON 리스트)
          "test_acc", "test_loss"]


def lr_str(lr):
    return f"{lr:.0e}".replace("e-0", "e-")      # 0.0001 -> 1e-4


def done_runs():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS, newline="", encoding="utf-8") as f:
        return {(r["framework"], r["optimizer"], float(r["lr"]), int(r["seed"]))
                for r in csv.DictReader(f)}


def save_row(row):
    is_new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)


def make_row(framework, optimizer, lr, seed, n_params, minutes, hist, test_loss, test_acc):
    return dict(framework=framework, optimizer=optimizer, lr=lr, seed=seed,
                n_params=n_params, minutes=round(minutes, 2),
                train_acc=json.dumps(hist["train_acc"]),
                train_loss=json.dumps(hist["train_loss"]),
                val_acc=json.dumps(hist["val_acc"]),
                val_loss=json.dumps(hist["val_loss"]),
                test_acc=test_acc, test_loss=test_loss)


def tag(framework, optimizer, lr, seed):
    return f"[{framework} · {optimizer} · lr {lr_str(lr)} · 시드 {seed}]"


def report(row):
    tr = json.loads(row["train_acc"])[-1]
    va = json.loads(row["val_acc"])[-1]
    diverged = not np.isfinite(json.loads(row["val_loss"])[-1])
    print(f"{tag(row['framework'], row['optimizer'], float(row['lr']), row['seed'])} "
          f"train {tr:.3f} · val {va:.3f} · test {float(row['test_acc']):.3f} "
          f"· {row['minutes']:.1f}분" + (" · 발산(loss NaN)" if diverged else ""))


def plan():
    done = done_runs()
    print("[계획] 후보 4개 × 시드 " + ", ".join(map(str, SEEDS))
          + " — CSV에 이미 있는 실행은 건너뛴다")
    for fw, name in (("keras", "Keras"), ("torch", "PyTorch")):
        todo = [1 for s in SEEDS for o, lr in CANDIDATES if (fw, o, lr, s) not in done]
        have = len(SEEDS) * len(CANDIDATES) - len(todo)
        print(f"  {name:8s} 이미 {have}번 · 남은 {len(todo)}번 ≈ {len(todo) * MIN_PER_RUN[fw]:.0f}분")
    if not os.path.exists(RESULTS):
        print("! Drive에 food11_v2_lr.csv가 없다 — 2단계 결과를 이 경로에 두지 않으면 시드 0도 새로 돈다")


plan()

## 셀 2 — Keras 8번 (후보 4개 × 시드 1·2) · 약 20분

In [ ]:
import keras
from keras import layers


def make_optimizer(name, lr):
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "sgd_momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    return keras.optimizers.SGD(learning_rate=lr)


def show_epoch(epoch, logs):
    print(f"  에폭 {epoch + 1}/{EPOCHS}  "
          f"train {logs['accuracy']:.3f} (loss {logs['loss']:.3f})  ·  "
          f"val {logs['val_accuracy']:.3f} (loss {logs['val_loss']:.3f})", flush=True)


def run_keras(opt_name, lr, seed):
    keras.utils.set_random_seed(seed)

    def load(split, shuffle):
        ds = keras.utils.image_dataset_from_directory(
            f"{DATA_DIR}/{split}", image_size=(IMG, IMG),
            batch_size=BATCH, shuffle=shuffle, seed=seed)
        rescale = layers.Rescaling(1.0 / 255)
        return ds.map(lambda x, y: (rescale(x), y))

    train_ds = load("training", True)
    val_ds = load("validation", False)
    test_ds = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = keras.Sequential([
        keras.Input(shape=(IMG, IMG, 3)),

        layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax",
                     kernel_initializer="he_normal"),
    ])
    n_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_params == N_PARAMS, n_params

    model.compile(
        optimizer=make_optimizer(opt_name, lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    t0 = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0,
                  callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=show_epoch)]).history
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    hist = {"train_acc": h["accuracy"], "train_loss": h["loss"],
            "val_acc": h["val_accuracy"], "val_loss": h["val_loss"]}
    row = make_row("keras", opt_name, lr, seed, n_params, (time.time() - t0) / 60,
                   {k: [float(v) for v in vs] for k, vs in hist.items()},
                   float(test_loss), float(test_acc))
    keras.backend.clear_session()
    gc.collect()
    return row


for seed in SEEDS:
    for opt_name, lr in CANDIDATES:
        if ("keras", opt_name, lr, seed) in done_runs():
            print(f"{tag('keras', opt_name, lr, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('keras', opt_name, lr, seed)} 시작 (약 2.5분)", flush=True)
        row = run_keras(opt_name, lr, seed)
        save_row(row)
        report(row)

## 셀 3 — PyTorch 8번 (후보 4개 × 시드 1·2) · 약 36분

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

tf_ = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),          # 0~1 변환 + HWC -> CHW
])


def make_optimizer(name, params, lr):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    return torch.optim.SGD(params, lr=lr)


def run_torch(opt_name, lr, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g = torch.Generator().manual_seed(seed)

    def load(split, shuffle):
        ds = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform=tf_)
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                          generator=g if shuffle else None,
                          num_workers=2, pin_memory=True)

    train_dl = load("training", True)
    val_dl = load("validation", False)
    test_dl = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Dropout(0.5),
        nn.Linear(64 * 30 * 30, NUM_CLASSES),
    ).to(device)

    for m in model:       # 1단계 — He 초기화. 가중치만, bias는 PyTorch 기본값 그대로
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_params == N_PARAMS, n_params

    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_optimizer(opt_name, model.parameters(), lr)

    def evaluate(dataloader):
        model.eval()
        total, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                pred = model(X)
                total += loss_fn(pred, y).item() * y.size(0)
                correct += (pred.argmax(1) == y).sum().item()
                n += y.size(0)
        return total / n, correct / n

    hist = {"train_acc": [], "train_loss": [], "val_acc": [], "val_loss": []}
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        total, correct, n = 0.0, 0, 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total += loss.item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, val_acc = evaluate(val_dl)
        hist["train_acc"].append(correct / n)
        hist["train_loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        hist["val_loss"].append(val_loss)
        print(f"  에폭 {epoch + 1}/{EPOCHS}  "
              f"train {hist['train_acc'][-1]:.3f} (loss {hist['train_loss'][-1]:.3f})  ·  "
              f"val {val_acc:.3f} (loss {val_loss:.3f})", flush=True)

    test_loss, test_acc = evaluate(test_dl)
    row = make_row("torch", opt_name, lr, seed, n_params, (time.time() - t0) / 60,
                   hist, test_loss, test_acc)
    del model, optimizer, train_dl, val_dl, test_dl
    torch.cuda.empty_cache()
    gc.collect()
    return row


for seed in SEEDS:
    for opt_name, lr in CANDIDATES:
        if ("torch", opt_name, lr, seed) in done_runs():
            print(f"{tag('torch', opt_name, lr, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('torch', opt_name, lr, seed)} 시작 (약 4.5분)", flush=True)
        row = run_torch(opt_name, lr, seed)
        save_row(row)
        report(row)

## 셀 4 — 요약과 결론 (이 셀의 출력을 붙여 주면 된다)

In [ ]:
import unicodedata

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

with open(RESULTS, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
assert rows, "결과가 아직 없다 — 셀 2·3을 먼저 돌릴 것"
assert all(int(r["n_params"]) == N_PARAMS for r in rows), "파라미터 수가 다른 실행이 있다"

FW = {"keras": "Keras", "torch": "PyTorch"}
OPT_NAME = {"adam": "Adam", "sgd_momentum": "SGD+momentum", "sgd": "SGD"}
OPT_SHORT = {"adam": "Adam", "sgd_momentum": "SGD+m", "sgd": "SGD"}
MIN_SEEDS = 3                  # 규칙을 적용하려면 조건마다 이만큼의 시드가 필요하다
A_NEW, A_BASE = ("adam", 1e-4), ("adam", 1e-3)      # 규칙 1: 옮길 후보와 현재 기준선
SGDM, SGD = ("sgd_momentum", 1e-3), ("sgd", 1e-2)
assert {A_NEW, A_BASE, SGDM, SGD} == set(CANDIDATES)


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def pad(s, w):
    return s + " " * max(0, w - width(s))


def bar(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def label(c):
    return f"{OPT_NAME[c[0]]} · {lr_str(c[1])}"


def short(c):
    return f"{OPT_SHORT[c[0]]} {lr_str(c[1])}"


def metrics(r):
    ta, tl = np.array(json.loads(r["train_acc"])), np.array(json.loads(r["train_loss"]))
    va, vl = np.array(json.loads(r["val_acc"])), np.array(json.loads(r["val_loss"]))
    ok = np.isfinite(vl)
    best = int(np.argmin(np.where(ok, vl, np.inf)))
    return dict(train_acc=ta[-1] * 100, val_acc=va[-1] * 100,
                best_val_loss=float(vl[best]) if ok.any() else np.nan,
                best_ep=best + 1 if ok.any() else np.nan,
                test_acc=float(r["test_acc"]) * 100, test_loss=float(r["test_loss"]),
                diverged=bool(not np.isfinite(vl[-1]) or not np.isfinite(tl[-1])))


runs = {}      # (프레임워크, 옵티마이저, 학습률) -> {시드: 지표}
for r in rows:
    c = (r["optimizer"], float(r["lr"]))
    if c in CANDIDATES:        # 2단계의 나머지 20칸(시드 0만 있음)은 여기서 쓰지 않는다
        runs.setdefault((r["framework"],) + c, {})[int(r["seed"])] = metrics(r)

fws = [fw for fw in FW if any(k[0] == fw for k in runs)]
assert fws, "후보 4개의 결과가 CSV에 없다 — 셀 1의 [계획]을 확인할 것"


def vals(group, key):
    v = np.array([m[key] for m in group.values()], dtype=float)
    return v[np.isfinite(v)]


def span(group, key, nd):
    v = vals(group, key)
    if len(v) == 0:
        return "-"
    if len(group) == 1:
        return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f} [{v.min():.{nd}f}~{v.max():.{nd}f}]"


def mean_of(group, key):
    v = vals(group, key)
    return float(v.mean()) if len(v) else np.nan


def notes(group):
    out = []
    n_div = sum(m["diverged"] for m in group.values())
    if n_div:
        out.append(f"발산 {n_div}/{len(group)}")
    live = [m for m in group.values() if not m["diverged"]]
    if live and all(m["best_ep"] == EPOCHS for m in live):
        out.append("최저 에폭=마지막")
    return " · ".join(out)


# ---- 결과표 -----------------------------------------------------------------
COLS = [("train_acc", "train acc %", 1), ("val_acc", "val acc %", 1),
        ("best_val_loss", "val loss 최저", 2), ("best_ep", "최저 에폭", 1),
        ("test_acc", "test acc %", 1)]
CW = 18
for fw in fws:
    bar(f"[결과표] {FW[fw]} — 후보 4개, 5에폭 끝 값, 평균 [최소~최대]")
    print(pad("옵티마이저 · lr", 22) + pad("시드", 8)
          + "".join(pad(name, CW) for _, name, _ in COLS) + "비고")
    for c in CANDIDATES:
        g = runs.get((fw,) + c)
        if not g:
            print(pad(label(c), 22) + "결과 없음")
            continue
        print(pad(label(c), 22) + pad(",".join(map(str, sorted(g))), 8)
              + "".join(pad(span(g, key, nd), CW) for key, _, nd in COLS) + notes(g))


# ---- 같은 시드끼리 비교 -----------------------------------------------------
def paired(fw, a, b, key, higher_better):
    """같은 시드끼리 뺀 값 a-b로 a와 b를 비교한다. higher_better=False면 값이 낮은 쪽이 낫다."""
    ga, gb = runs.get((fw,) + a, {}), runs.get((fw,) + b, {})
    seeds = [s for s in sorted(set(ga) & set(gb))
             if np.isfinite(ga[s][key]) and np.isfinite(gb[s][key])]
    out = dict(diffs=[], mean=np.nan, limit=np.nan, winner=None, enough=len(seeds) >= MIN_SEEDS,
               text=f"판정 안 함 (공통 시드 {len(seeds)}개, {MIN_SEEDS}개 필요)")
    if not out["enough"]:
        return out
    va = np.array([ga[s][key] for s in seeds])
    vb = np.array([gb[s][key] for s in seeds])
    d = va - vb
    gain = d if higher_better else -d              # 양수 = a가 낫다
    limit = float(max(np.ptp(va), np.ptp(vb)))      # 시드 범위: 두 조건 중 큰 쪽
    out.update(diffs=[float(x) for x in d], mean=float(d.mean()), limit=limit)
    consistent = bool((gain > 0).all() or (gain < 0).all())
    if consistent and abs(float(gain.mean())) > limit:
        out["winner"] = "a" if gain.mean() > 0 else "b"
        out["text"] = f"{short(a if out['winner'] == 'a' else b)} 우세 (뚜렷함)"
    elif not consistent:
        out["text"] = (f"구분 안 됨 — 시드마다 방향이 다름 "
                       f"({short(a)} 우세 {int((gain > 0).sum())}/{len(seeds)})")
    else:
        out["text"] = "구분 안 됨 — 방향은 같지만 평균 차이가 시드 범위 안"
    return out


def show_cmp(fw, a, b, key, higher_better, title, tw, nd, unit):
    c = paired(fw, a, b, key, higher_better)
    head = pad(FW[fw], 9) + pad(title, tw)
    if not c["diffs"]:
        print(head + c["text"])
    else:
        per = " ".join(f"{x:+.{nd}f}" for x in c["diffs"])
        print(head + f"{c['mean']:+.{nd}f}{unit}  (시드별 {per})  시드 범위 {c['limit']:.{nd}f}"
              + f"  -> {c['text']}")
    return c


# ---- 학습률 결론 1: Adam 기준선을 옮길까 ------------------------------------
bar("[학습률 결론 1] Adam 기준선을 1e-3에서 1e-4로 옮길까 — 결과를 보기 전에 정한 규칙\n"
    "   옮기는 조건: 아래 4칸(val acc · val loss 최저 × 두 프레임워크) 전부에서 1e-4가 '뚜렷하게' 앞설 것\n"
    "   뚜렷함 = 같은 시드끼리 뺀 값이 시드 전부 같은 방향이고, 평균 차이가 시드 범위"
    "(두 조건의 최대−최소 중 큰 쪽)보다 클 것\n"
    f"   차이 = {label(A_NEW)} − {label(A_BASE)}  (val loss는 낮은 쪽이 낫다)")
cells = []
for fw in fws:
    cells.append(show_cmp(fw, A_NEW, A_BASE, "val_acc", True, "val acc", 16, 1, "%p"))
    cells.append(show_cmp(fw, A_NEW, A_BASE, "best_val_loss", False, "val loss 최저", 16, 2, ""))
n_new = sum(c["winner"] == "a" for c in cells)
n_base = sum(c["winner"] == "b" for c in cells)
n_skip = sum(not c["enough"] for c in cells)              # 시드가 모자라 판정하지 못한 칸
n_tie = len(cells) - n_new - n_base - n_skip
print()
if len(fws) < 2 or not all(c["enough"] for c in cells):
    verdict = "판정 보류 — 두 프레임워크 모두 시드 3개씩의 결과가 필요하다 (결과가 더 쌓이면 다시 돌릴 것)"
elif n_new == len(cells):
    verdict = "기준선을 Adam 1e-4로 옮긴다 (4칸 전부에서 1e-4가 뚜렷하게 앞섰다)"
elif n_base == len(cells):
    verdict = "Adam 1e-3 확정 (4칸 전부에서 1e-3이 뚜렷하게 앞섰다)"
else:
    verdict = "Adam 1e-3 유지 — 1e-4와 구분되지 않는다"
print(f"-> 결론: {verdict}")
print(f"   (1e-4 우세 {n_new}/{len(cells)}칸 · 1e-3 우세 {n_base}/{len(cells)}칸 · "
      f"구분 안 됨 {n_tie}칸" + (f" · 판정 안 함 {n_skip}칸" if n_skip else "") + ")")
print("   참고: Adam 1e-4보다 낮은 학습률(3e-5 등)은 이번에도 보지 않았다 (격자 끝)")

# ---- 옵티마이저 비교 --------------------------------------------------------
bar("[옵티마이저 비교] 옵티마이저마다 최적 후보끼리, val acc 기준 (test는 보고용)\n"
    "   Adam은 두 후보(1e-4 · 1e-3) 중 3시드 평균 val acc가 높은 쪽. 판정 기준은 위와 같다\n"
    "   차이 = 앞쪽 − 뒤쪽")


def score(fw, c):
    m = mean_of(runs.get((fw,) + c, {}), "val_acc")
    return -np.inf if np.isnan(m) else m


for fw in fws:
    adam_c = max((A_NEW, A_BASE), key=lambda c: score(fw, c))
    for a, b in ((adam_c, SGDM), (adam_c, SGD), (SGDM, SGD)):
        show_cmp(fw, a, b, "val_acc", True, f"{short(a)} − {short(b)}", 26, 1, "%p")

# ---- 그래프 -----------------------------------------------------------------
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
OPT_STYLE = {"adam": ("#2a78d6", "o"), "sgd_momentum": ("#eb6834", "s"), "sgd": ("#1baf7a", "^")}
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


def style(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)


PANELS = [("val_acc", "val acc (%) at epoch 5", 1.0),
          ("best_val_loss", "best val loss (min over epochs)", 0.03)]
fig, axes = plt.subplots(len(fws), 2, figsize=(10.5, 3.7 * len(fws) + 0.9), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (key, name, min_pad) in enumerate(PANELS):
        ax = axes[i][j]
        style(ax)
        seen = []
        for x, c in enumerate(CANDIDATES):
            color, marker = OPT_STYLE[c[0]]
            pts = [(s, m[key]) for s, m in sorted(runs.get((fw,) + c, {}).items())
                   if np.isfinite(m[key])]
            if not pts:
                continue
            ys = np.array([p[1] for p in pts])
            seen += list(ys)
            ax.vlines(x, ys.min(), ys.max(), color=color, lw=1.6, alpha=0.45, zorder=2)
            ax.plot([x - 0.3, x + 0.3], [ys.mean()] * 2, color=color, lw=2.4,
                    solid_capstyle="butt", zorder=3)
            ax.plot([x + 0.2 * (s - 1) for s, _ in pts], ys, ls="none", marker=marker, ms=6.5,
                    mfc=color, mec=SURFACE, mew=1.1, zorder=4)
        if seen:
            lo, hi = min(seen), max(seen)
            margin = max(min_pad, (hi - lo) * 0.2)
            ax.set_ylim(lo - margin, hi + margin)
        ax.set_xlim(-0.6, len(CANDIDATES) - 0.4)
        ax.set_xticks(range(len(CANDIDATES)))
        ax.set_xticklabels([f"{OPT_SHORT[c[0]]}\n{lr_str(c[1])}" + (" (base)" if c == A_BASE else "")
                            for c in CANDIDATES])
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=OPT_STYLE[o][0], marker=OPT_STYLE[o][1], ls="none", ms=6.5,
                  mec=SURFACE, mew=1.1, label=OPT_NAME[o])
           for o in OPT_NAME if any(c[0] == o for c in CANDIDATES)]
handles += [Line2D([0], [0], color=INK2, lw=2.4, label="mean"),
            Line2D([0], [0], color=INK2, marker="o", ls="none", ms=5,
                   label="each seed (left to right: 0, 1, 2)")]
fig.text(0.005, 0.99, "Learning-rate step, 3-seed check  -  He init, 5 epochs",
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("lr_seeds.png", facecolor=SURFACE)
plt.show()
print("[그래프] lr_seeds.png")